### MOUNT GOOGLE DRIVE

In [1]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Google Drive sudah ter-mount di /content/drive")

Google Drive sudah ter-mount di /content/drive


### KONFIGURASI

In [2]:
CONFIG = {
    "dataset_path": "/content/drive/MyDrive/PDP_OCR_2/dataset",
    "result_path": "/content/drive/MyDrive/PDP_OCR_2/exp6_paddleV6CRAFT/result_v6Craft",
    "supported_ext": (".jpg", ".jpeg", ".png"),
    "start_index": 0,
    "limit": None,
    "gt_only": True,
    "gt_path": "/content/drive/MyDrive/PDP_OCR_2/groundtruth",
    "gt_extension": ".md",
    "lang": "en",
    "device": "gpu:0",
    "use_doc_orientation_classify": False,
    "use_doc_unwarping": False,
    "use_textline_orientation": True,
    "text_det_thresh": 0.3,
    "text_det_box_thresh": 0.6,
    "text_det_unclip_ratio": 1.5,
    "text_rec_score_thresh": 0.0,
}

In [3]:
import os
os.makedirs(CONFIG["result_path"], exist_ok=True)

### Instalasi PaddleOCR (Recognition) + CRAFT (Detection)

In [4]:
%%bash
nvidia-smi --query-gpu=name,driver_version --format=csv,noheader 2>/dev/null || echo
python -c "import torch; print('CUDA runtime:', torch.version.cuda)" 2>/dev/null
if command -v nvidia-smi &> /dev/null && nvidia-smi -L &> /dev/null; then
    echo "GPU terdeteksi -> install paddlepaddle-gpu"
    pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
else
    echo "GPU tidak terdeteksi -> install paddlepaddle (CPU)"
    pip install -q paddlepaddle
fi

pip install -q "paddleocr>=3.0.0"
pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --upgrade
pip install -q --no-deps craft-text-detector
pip install -q gdown


CUDA runtime: None
GPU tidak terdeteksi -> install paddlepaddle (CPU)


### IMPORT LIBRARY

In [5]:
import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import time
import cv2
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image, ImageDraw
if 'TextRecognition' not in globals():
    from paddleocr import TextRecognition

### MONKEY PATCHES UNTUK DEPENDENSI INTERNAL (CRAFT & TORCHVISION)

In [6]:
import collections
import collections.abc
for _name in ("Mapping", "MutableMapping", "Sequence", "Iterable"):
    if not hasattr(collections, _name):
        setattr(collections, _name, getattr(collections.abc, _name))

import torch
import torchvision.models as _tv_models
import torchvision.models.vgg as _tv_vgg

if not getattr(_tv_vgg, "_craft_patch_applied", False):
    if not hasattr(_tv_vgg, "model_urls"):
        _tv_vgg.model_urls = {
            "vgg11": "https://download.pytorch.org/models/vgg11-8a719046.pth",
            "vgg13": "https://download.pytorch.org/models/vgg13-19584684.pth",
            "vgg16": "https://download.pytorch.org/models/vgg16-397923af.pth",
            "vgg19": "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth",
            "vgg11_bn": "https://download.pytorch.org/models/vgg11_bn-6002323d.pth",
            "vgg13_bn": "https://download.pytorch.org/models/vgg13_bn-abd245e5.pth",
            "vgg16_bn": "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth",
            "vgg19_bn": "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth",
        }

    _true_orig_vgg16_bn = _tv_vgg.vgg16_bn


    def _patched_vgg16_bn(pretrained=False, progress=True, **kwargs):
        kwargs.pop("weights", None)
        try:
            return _true_orig_vgg16_bn(
                weights="IMAGENET1K_V1" if pretrained else None,
                progress=progress,
                **kwargs,
            )
        except TypeError:
            return _true_orig_vgg16_bn(pretrained=pretrained, progress=progress, **kwargs)

    _tv_models.vgg16_bn = _patched_vgg16_bn
    _tv_vgg.vgg16_bn = _patched_vgg16_bn
    _tv_vgg._craft_patch_applied = True

if not getattr(torch, "_craft_load_patch_applied", False):
    _orig_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _orig_torch_load(*args, **kwargs)
    torch.load = _patched_torch_load
    torch._craft_load_patch_applied = True

if 'Craft' not in globals():
    from craft_text_detector import Craft

import contextlib

@contextlib.contextmanager
def numpy_allow_ragged():
    _orig_array = np.array
    def _patched_array(obj, *args, **kwargs):
        try:
            return _orig_array(obj, *args, **kwargs)
        except ValueError as e:
            if "inhomogeneous" in str(e) or "sequence" in str(e):
                kwargs["dtype"] = object
                return _orig_array(obj, *args, **kwargs)
            raise
    np.array = _patched_array
    try:
        yield
    finally:
        np.array = _orig_array

### INISIALISASI MODEL

In [7]:
# ---- CRAFT (deteksi bounding box) ----
_craft_use_cuda = CONFIG["device"].startswith("gpu") and torch.cuda.is_available()
if CONFIG["device"].startswith("gpu") and not torch.cuda.is_available():
    print("[INFO] CONFIG['device'] diset ke GPU tapi CUDA tidak tersedia di runtime ini, CRAFT dijalankan di CPU.")

if 'craft' not in globals():
    craft = Craft(
        output_dir=None,
        crop_type="box",
        cuda=_craft_use_cuda,
        text_threshold=CONFIG["text_det_thresh"],
        low_text=CONFIG["text_det_box_thresh"],
    )

def build_text_recognizer(device):
    try:
        return TextRecognition(model_name="PP-OCRv6_medium_rec", device=device)
    except Exception:

        return TextRecognition(device=device)

if 'text_recognizer' not in globals():
    try:
        text_recognizer = build_text_recognizer(CONFIG["device"])
    except Exception as e:
        print(f"GPU gagal ({e}), fallback ke CPU")
        text_recognizer = build_text_recognizer("cpu")

[INFO] CONFIG['device'] diset ke GPU tapi CUDA tidak tersedia di runtime ini, CRAFT dijalankan di CPU.


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
The specified device (GPU) is not available! Switching to CPU instead.


### HELPER FUNCTION

In [8]:
def draw_boxes(image, boxes, texts=None, scores=None):
    img = image.copy()
    draw = ImageDraw.Draw(img)
    for i, box in enumerate(boxes):
        pts = np.array(box).astype(np.int32).reshape(-1, 2)
        draw.polygon([tuple(p) for p in pts], outline="red", width=3)
        if texts is not None:
            label = texts[i] if scores is None else f"{texts[i]} ({scores[i]:.2f})"
            draw.text((pts[0][0], max(pts[0][1] - 15, 0)), label, fill="blue")
    return img


def show_image(img, title):
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')
    plt.show()


def crop_polygon(image_np, box):
    pts = np.array(box).astype(np.float32)
    x_min = max(int(np.min(pts[:, 0])), 0)
    y_min = max(int(np.min(pts[:, 1])), 0)
    x_max = min(int(np.max(pts[:, 0])), image_np.shape[1])
    y_max = min(int(np.max(pts[:, 1])), image_np.shape[0])
    crop = image_np[y_min:y_max, x_min:x_max]
    if crop.size == 0:
        return crop
    return cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)


def reconstruct_indentation(boxes, texts, tolerance=45, use_indent=True):
    if not boxes or not texts:
        return ""

    data = []
    for box, text in zip(boxes, texts):
        x_coords = [pt[0] for pt in box]
        y_coords = [pt[1] for pt in box]
        x_min = min(x_coords)
        y_center = sum(y_coords) / 4.0
        h = max(y_coords) - min(y_coords)
        data.append({"x": x_min, "y": y_center, "h": h, "text": text})

    data.sort(key=lambda item: item["y"])

    lines = []
    current_line = [data[0]]

    for item in data[1:]:
        prev_item = current_line[-1]
        if abs(item["y"] - prev_item["y"]) < max(item["h"], prev_item["h"]) * 0.5:
            current_line.append(item)
        else:
            lines.append(current_line)
            current_line = [item]
    lines.append(current_line)

    first_x_list = sorted([line[0]["x"] for line in lines])

    clusters = []
    if first_x_list:
        current_cluster = [first_x_list[0]]
        for x in first_x_list[1:]:
            if x - current_cluster[0] <= tolerance:
                current_cluster.append(x)
            else:
                clusters.append(current_cluster)
                current_cluster = [x]
        clusters.append(current_cluster)

    def get_indent_level(x_val):
        for i, cluster in enumerate(clusters):
            if x_val in cluster:
                return i
        return 0

    result_lines = []
    for line in lines:
        line.sort(key=lambda item: item["x"])
        first_x = line[0]["x"]

        level = get_indent_level(first_x) if use_indent else 0

        line_str = "    " * level + line[0]["text"]

        for i in range(1, len(line)):
            line_str += " " + line[i]["text"]

        result_lines.append(line_str)

    return "\n".join(result_lines)


def write_raw_markdown(image_name, text_content):
    md_path = os.path.join(CONFIG["result_path"], os.path.splitext(image_name)[0] + ".md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(text_content)

### MAIN FUNCTION

In [9]:
def process_ocr(image_name, index, total):
    img_path = os.path.join(CONFIG["dataset_path"], image_name)

    start = time.perf_counter()

    image = Image.open(img_path).convert('RGB')
    image_np = np.array(image)

    # STEP 0: Deteksi bounding box dengan CRAFT (menggantikan detektor bawaan PaddleOCR)
    with numpy_allow_ragged():
        craft_result = craft.detect_text(image=img_path)
    # PENTING: konversi ke list Python biasa. numpy_allow_ragged() bisa
    # mengembalikan boxes sebagai numpy array dtype=object, dan itu bikin
    # `if not boxes` di reconstruct_indentation() error ("truth value of an
    # array... is ambiguous") karena numpy array tidak bisa dicek truthy
    # langsung kalau isinya lebih dari 1 elemen.
    boxes = list(craft_result["boxes"])

    # STEP 0b: Baca teks tiap box pakai PP-OCRv6 (recognition-only)
    texts, scores = [], []
    for box in boxes:
        crop = crop_polygon(image_np, box)
        if crop.size == 0:
            texts.append("")
            scores.append(0.0)
            continue
        rec_output = list(text_recognizer.predict(input=crop))
        if rec_output:
            texts.append(rec_output[0]["rec_text"])
            scores.append(rec_output[0]["rec_score"])
        else:
            texts.append("")
            scores.append(0.0)

    # Filter berdasarkan skor recognition (setara text_rec_score_thresh di exp5)
    if CONFIG["text_rec_score_thresh"] > 0:
        filtered = [
            (b, t, s) for b, t, s in zip(boxes, texts, scores)
            if s >= CONFIG["text_rec_score_thresh"]
        ]
        boxes = [f[0] for f in filtered]
        texts = [f[1] for f in filtered]
        scores = [f[2] for f in filtered]

    elapsed = time.perf_counter() - start

    print(f"\n[{index}/{total}] {image_name} — {elapsed:.2f}s")
    print("=" * 50)

    if len(boxes) == 0:
        print("Tidak ada teks terdeteksi")
        return

    # STEP 1: Bounding Box (CRAFT)
    box_only = draw_boxes(image, boxes)
    print(f"Step 1: Deteksi bounding box selesai ({len(boxes)} box) [CRAFT]")
    show_image(box_only, f"{image_name} — Step 1: Bounding Box (CRAFT)")

    # STEP 2: Hasil Recognition (Raw Text) [PP-OCRv6]
    print("\nStep 2: Hasil Recognition (Raw Text) [PP-OCRv6]")
    print("-" * 30)
    for i, t in enumerate(texts, 1):
        print(f"{i}. {t}")

    # STEP 3: Post-Processing Indentasi + Pengecualian
    base_name = os.path.splitext(image_name)[0]
    # Cek apakah nama file berakhiran '4_jawaban' atau 'penjelasan'
    if base_name.endswith("4_jawaban") or base_name.endswith("penjelasan"):
        use_indent = False
        print("\n[INFO] File terdeteksi narasi/penjelasan. Indentasi dinonaktifkan (Rata Kiri).")
    else:
        use_indent = True

    print("\nStep 3: Hasil Rekonstruksi Teks")
    print("-" * 30)
    reconstructed_text = reconstruct_indentation(boxes, texts, tolerance=45, use_indent=use_indent)
    print(reconstructed_text)
    print("=" * 50)

    # STEP 4: Simpan file markdown mentahan
    write_raw_markdown(image_name, reconstructed_text)
    print(f"File mentahan tersimpan: {base_name}.md")

### EKSEKUSI PIPELINE

In [10]:
def has_gt(filename):
    base = os.path.splitext(filename)[0]
    return os.path.exists(os.path.join(CONFIG["gt_path"], base + CONFIG["gt_extension"]))


if os.path.exists(CONFIG["dataset_path"]):
    files = sorted(f for f in os.listdir(CONFIG["dataset_path"]) if f.lower().endswith(CONFIG["supported_ext"]))

    if CONFIG["gt_only"]:
        files = [f for f in files if has_gt(f)]

    start = CONFIG["start_index"]
    end = start + CONFIG["limit"] if CONFIG["limit"] else None
    files = files[start:end]

    total = len(files)
    print(f"Total foto yang akan diproses: {total}")
    print(f"Rentang index: {start} - {start + total - 1}")
    print(f"GT only: {CONFIG['gt_only']}")
    print("Daftar file:")
    for i, f in enumerate(files, start + 1):
        print(f"  [{i}] {f}")
    print("-" * 40)

    for idx, file in enumerate(files, 1):
        try:
            process_ocr(file, idx, total)
        except Exception as e:
            print(f"Skip {file}: {e}")
else:
    print(f"Dataset path not found: {CONFIG['dataset_path']}")

Output hidden; open in https://colab.research.google.com to view.

### BENCHMARK TEST

In [11]:
import os
import re

def levenshtein_dist(seq1, seq2):
    if len(seq1) < len(seq2):
        return levenshtein_dist(seq2, seq1)
    if len(seq2) == 0:
        return len(seq1)

    prev_row = range(len(seq2) + 1)
    for i, c1 in enumerate(seq1):
        curr_row = [i + 1]
        for j, c2 in enumerate(seq2):
            ins = prev_row[j + 1] + 1
            it_del = curr_row[j] + 1
            sub = prev_row[j] + (c1 != c2)
            curr_row.append(min(ins, it_del, sub))
        prev_row = curr_row
    return prev_row[-1]

def clean_text_norm(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def calculate_metrics(gt_text, ocr_text):
    # --- STANDARD (Raw) ---
    gt_std = gt_text.strip()
    ocr_std = ocr_text.strip()

    cer_std = levenshtein_dist(gt_std, ocr_std) / max(1, len(gt_std))
    wer_std = levenshtein_dist(gt_std.split(), ocr_std.split()) / max(1, len(gt_std.split()))

    # --- NORMALIZED ---
    gt_norm = clean_text_norm(gt_text)
    ocr_norm = clean_text_norm(ocr_text)

    cer_norm = levenshtein_dist(gt_norm, ocr_norm) / max(1, len(gt_norm))
    wer_norm = levenshtein_dist(gt_norm.split(), ocr_norm.split()) / max(1, len(gt_norm.split()))

    # --- KEYWORD ACCURACY (KW Acc) ---
    gt_words = set([w for w in gt_norm.split() if len(w) > 2])
    if gt_words:
        ocr_words = set(ocr_norm.split())
        matched_words = gt_words.intersection(ocr_words)
        kw_acc = len(matched_words) / len(gt_words)
    else:
        kw_acc = 1.0

    return cer_std, cer_norm, wer_std, wer_norm, kw_acc

# Jalankan evaluasi jika folder GT ada
if os.path.exists(CONFIG["gt_path"]) and os.path.exists(CONFIG["result_path"]):
    md_files = [f for f in os.listdir(CONFIG["result_path"]) if f.endswith(".md")]
    md_files = sorted(md_files)

    print("| File Name | CER Std | CER Norm | WER Std | WER Norm | KW Acc |")
    print("|---|---|---|---|---|---|")

    sum_cer_std, sum_cer_norm, sum_wer_std, sum_wer_norm, sum_kw_acc = 0, 0, 0, 0, 0
    valid_count = 0

    for md_file in md_files:
        base_name = os.path.splitext(md_file)[0]
        gt_file = base_name + CONFIG["gt_extension"]
        gt_full_path = os.path.join(CONFIG["gt_path"], gt_file)
        md_full_path = os.path.join(CONFIG["result_path"], md_file)

        if os.path.exists(gt_full_path):
            with open(gt_full_path, "r", encoding="utf-8") as f:
                gt_content = f.read()
            with open(md_full_path, "r", encoding="utf-8") as f:
                ocr_content = f.read()

            c_std, c_norm, w_std, w_norm, kw = calculate_metrics(gt_content, ocr_content)

            print(f"| {md_file} | {c_std*100:.2f}% | {c_norm*100:.2f}% | {w_std*100:.2f}% | {w_norm*100:.2f}% | {kw*100:.1f}% |")

            sum_cer_std += c_std
            sum_cer_norm += c_norm
            sum_wer_std += w_std
            sum_wer_norm += w_norm
            sum_kw_acc += kw
            valid_count += 1

    if valid_count > 0:
        avg_cer_std = (sum_cer_std / valid_count) * 100
        avg_cer_norm = (sum_cer_norm / valid_count) * 100
        avg_wer_std = (sum_wer_std / valid_count) * 100
        avg_wer_norm = (sum_wer_norm / valid_count) * 100
        avg_kw_acc = (sum_kw_acc / valid_count) * 100

        print("|---|---|---|---|---|---|")
        print(
            f"| **AVERAGE ({valid_count} Files)** | "
            f"**{avg_cer_std:.2f}%** | "
            f"**{avg_cer_norm:.2f}%** | "
            f"**{avg_wer_std:.2f}%** | "
            f"**{avg_wer_norm:.2f}%** | "
            f"**{avg_kw_acc:.1f}%** |"
        )
    else:
        print("Tidak ditemukan file Ground Truth (.md) yang cocok di folder target.")

| File Name | CER Std | CER Norm | WER Std | WER Norm | KW Acc |
|---|---|---|---|---|---|
| if4901_103011500026_1a_jawaban.md | 8.67% | 2.66% | 31.37% | 15.38% | 89.5% |
| if4901_103011500026_1b_jawaban.md | 27.47% | 15.00% | 58.90% | 38.18% | 60.0% |
| if4901_103011500026_2a_jawaban.md | 6.26% | 1.83% | 21.43% | 12.73% | 91.4% |
| if4901_103011500026_2b_jawaban.md | 1.90% | 0.94% | 11.32% | 9.47% | 93.3% |
| if4901_103011500026_3_jawaban.md | 5.50% | 3.02% | 26.09% | 22.50% | 88.9% |
| if4901_103011500026_4_jawaban.md | 70.27% | 64.84% | 69.39% | 68.79% | 11.8% |
| if4901_103011500114_1a_jawaban.md | 54.97% | 48.94% | 82.61% | 75.00% | 53.1% |
| if4901_103011500114_1b_jawaban.md | 40.54% | 32.39% | 81.11% | 62.16% | 58.5% |
| if4901_103011500114_2a_jawaban.md | 8.12% | 1.97% | 27.07% | 18.64% | 87.1% |
| if4901_103011500114_2b_jawaban.md | 7.11% | 3.14% | 18.55% | 13.89% | 91.0% |
| if4901_103011500114_3_jawaban.md | 24.98% | 11.49% | 41.95% | 32.45% | 97.8% |
| if4901_103011500114_4